In [1]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [2]:
raw_inputs = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
]
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
print(inputs)

"""padding（填充）和 truncation（截断）；
可以传递一个句子或一组句子，还可以指定要返回的 tensor 类型
（如果没有传递类型，默认返回的是 python 中的 list 格式）。
"""

{'input_ids': tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102],
        [  101,  1045,  5223,  2023,  2061,  2172,   999,   102,     0,     0,
             0,     0,     0,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]])}


'padding（填充）和 truncation（截断）；\n可以传递一个句子或一组句子，还可以指定要返回的 tensor 类型\n（如果没有传递类型，默认返回的是 python 中的 list 格式）。\n'

输出是一个包含两个键， input_ids 和 attention_mask 。 input_ids 包含两行整数（每个句子一行），它们是每个句子中 token 的 ID。attention_mask 防止模型把补齐用的 padding 当成真实内容。

In [3]:
from transformers import AutoModel

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModel.from_pretrained(checkpoint)

In [4]:
outputs = model(**inputs)
print(outputs.last_hidden_state.shape)

torch.Size([2, 16, 768])


In [5]:
from transformers import AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
outputs = model(**inputs)

In [6]:
print(outputs.logits.shape)

torch.Size([2, 2])


现在输出的形状，其维度会降低很多：模型头接收我们之前看到的高维向量作为输入，并输出包含两个值（每种标签一个）的向量。
由于我们只有两个句子和两种标签，所以我们从模型中得到的结果的形状是 2 x 2。

In [7]:
print(outputs.logits)

tensor([[-1.5607,  1.6123],
        [ 4.1692, -3.3464]], grad_fn=<AddmmBackward0>)


[-1.5607, 1.6123]对第一句和[ 4.1692, -3.3464]对第二句进行预测。这些不是概率，而是logits，即模型最后一层输出的原始、未经归一化的分数。要将其转换为概率，需要经过一个SoftMax层（所有Transformer模型都输出logits，因为训练损失函数通常会将最后一个激活函数（例如SoftMax）与实际损失函数（例如交叉熵）融合）：

In [8]:
import torch

predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(predictions)

tensor([[4.0195e-02, 9.5980e-01],
        [9.9946e-01, 5.4418e-04]], grad_fn=<SoftmaxBackward0>)


现在我们可以看到，模型[0.0402, 0.9598]对第一句话和[0.9995,  0.0005]第二句话都做出了预测。这些都是可以识别的概率得分。要获取每个位置对应的标签，我们可以检查id2label模型配置的属性：

In [9]:
model.config.id2label

{0: 'NEGATIVE', 1: 'POSITIVE'}

现在我们可以得出结论，该模型预测了以下内容：

第一句：消极：0.0402，积极：0.9598
第二句：消极：0.9995，积极：0.0005